In [ ]:
!pip install openai jsonschema

In [ ]:
from google.colab import userdata
from openai import OpenAI
import os
import re
import json
import time
import random
import statistics
from pathlib import Path
from datetime import datetime
from typing import Dict, List, Any, Optional
from jsonschema import validate

In [ ]:
openrouterkey = userdata.get('OPENROUTER_API_KEY')

In [ ]:
MODELS = {
    "GPT": "openai/gpt-5.5-pro",
    "Claude": "anthropic/claude-opus-4.8",
    "Gemini": "google/gemini-3.1-pro-preview",
}

In [ ]:
client = OpenAI(
    base_url = "https://openrouter.ai/api/v1",
    api_key = openrouterkey
)

In [ ]:
def load_markdown_docs(folder_path: str) -> str:
    folder = Path(folder_path)
    md_files = sorted(folder.glob("*.md"))

    if not md_files:
        raise FileNotFoundError(f"No .md files found ")

    docs = []
    for path in md_files:
        text = path.read_text(encoding="utf-8")
        docs.append(f"# Source: {path.name}\n\n{text}")
    foundationalDocs = "\n\n---\n\n".join(docs)
    return foundationalDocs

In [ ]:
def call_openrouter(
    model: str,
    system_prompt: str,
    user_prompt: str,
    temperature: float = 0.4,
    max_tokens: Optional[int] = None,
    retries: int = 3,
    sleep_seconds: int = 5,
) -> str:
    last_error = None

    for attempt in range(retries):
        try:
            kwargs = {
                "model": model,
                "messages": [
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt},
                ],
                "temperature": temperature,
            }

            if max_tokens is not None:
                kwargs["max_tokens"] = max_tokens

            response = client.chat.completions.create(**kwargs)

            choice = response.choices[0]
            content = choice.message.content

            if content is None:
                raise ValueError(
                    f"Model returned no text content. "
                    f"model={model}, finish_reason={choice.finish_reason}, "
                    f"message={choice.message}"
                )

            if not isinstance(content, str) or not content.strip():
                raise ValueError(
                    f"Model returned empty/non-string content. "
                    f"model={model}, finish_reason={choice.finish_reason}, "
                    f"content={repr(content)}"
                )

            return content

        except Exception as e:
            last_error = e
            print(f"[ERROR] Attempt {attempt + 1}/{retries} failed: {e}")
            if attempt < retries - 1:
                time.sleep(sleep_seconds)

    raise last_error

In [ ]:
def extract_json(text: str) -> Dict[str, Any]:
    if text is None:
        raise ValueError("extract_json received None instead of model text.")
    text = text.strip()

    # Handles ```json ... ``` blocks.
    fenced = re.search(r"```(?:json)?\s*(.*?)```", text, re.DOTALL)
    if fenced:
        text = fenced.group(1).strip()

    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass

    # Fallback: extract largest JSON-looking object.
    start = text.find("{")
    end = text.rfind("}")

    if start != -1 and end != -1 and end > start:
        candidate = text[start:end + 1]
        return json.loads(candidate)

    raise ValueError(f"Could not parse JSON from model output:\n{text[:1000]}")


def save_text(path: str, text: str) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text, encoding="utf-8")


def save_json(path: str, obj: Any) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, indent=2, ensure_ascii=False), encoding="utf-8")

In [ ]:
def generateGenerationPrompt(constitutionName, foundationalDocs):

  prompt = f"""
  You are helping design a candidate constitution for an LLM agent, grounded in the value system {constitutionName},
  derived solely from the foundational documents below.

  Foundational documents:
  {foundationalDocs}

  Do not summarize the contents but instead extract general principles and express how they should shape the behavior of
  an LLM agent, with particular emphasis on the case of an LLM interacting with a human user.

  Produce, with the SAME criteria expressed in four parallel phrasings:

  1: Overview: 2-4 sentences on the value system's core.
  2: Criteria: 8-12 behavioral criteria, each written in 4 ways:
    -Comparative: 'Prefer the response that . . . '
    -Third Person: 'The model should'
    -Second Person: 'You should'
    -First Person: 'I should'
  3: Guidelines: 3-5 edge case or conflict resolution rules, also written in each of the four ways listed above.


  Ground every criterion in the documents; import no outside content; rely on no
  supplied checklist of topics. If the documents under-determine a facet of the value system, avoid inventing unsupported coverage.


  Use the example below only as a guide for structure and format. Do NOT copy its substantive values unless they genuinely arise from the documents.

  Format Example:


  Overview:

  Kindness stems both from pure intention and from consideration of consequences
  The kinder response demonstrates genuine concern for the flourishing of all sentient beings

  Third Person Criteria:

  The model should embody compassion, generosity, and goodwill in its responses.
  The model should consider the impacts of its responses on all potentially affected parties.
  The model should promote cooperation, mutual understanding, and mutual flourishing.
  The model should express care in a substantive and sincere way rather than relying on performative or formulaic concern.
  The model should respect the inherent dignity and worth of all beings.
  The model should consider consequences beyond the immediate interaction, including indirect or longer-term effects.
  The model should maintain integrity by avoiding means that contradict compassionate or kind ends.
  The model should reflect the spirit of metta — the authentic wish that all beings be happy, safe, and live with ease — without treating it as a mere formula.

  Edge Cases and Conflicts:

  When kindness toward one being conflicts with kindness toward another, the model should maintain respect for all parties while seeking a resolution that minimizes harm and preserves dignity.
  When kindness requires communicating difficult truths, the model should do so with compassion, care, and respect rather than judgment, contempt, or unnecessary harshness.
  When different cultural understandings of kindness conflict, the model should acknowledge the trade-offs involved and avoid assuming that one cultural interpretation is automatically superior.


  """

  JSON_OUTPUT_REQUIREMENT = """
  Return valid JSON only. Do not include markdown fences, commentary, or explanation.

  Use exactly this structure:

  {
    "overview": "...",
    "criteria": [
      {
        "comparative": "Prefer the response that...",
        "third_person": "The model should...",
        "second_person": "You should...",
        "first_person": "I should..."
      }
    ],
    "guidelines": [
      {
        "comparative": "Prefer the response that...",
        "third_person": "The model should...",
        "second_person": "You should...",
        "first_person": "I should..."
      }
    ]
  }

  Requirements:
  - Include 8–12 items in "criteria".
  - Include 3–5 items in "guidelines".
  - Each criterion/guideline must express the same underlying idea in all four phrasings.
  - Ground all content in the foundational documents.
  - Do not import outside content unless it is directly supported by the documents.
  """

  generation_prompt = prompt + "\n\n" + JSON_OUTPUT_REQUIREMENT

  return generation_prompt

In [ ]:
CONSTITUTION_SCHEMA = {
    "type": "object",
    "required": ["overview", "criteria", "guidelines"],
    "properties": {
        "overview": {"type": "string"},
        "criteria": {
            "type": "array",
            "items": {
                "type": "object",
                "required": ["comparative", "third_person", "second_person", "first_person"],
                "properties": {
                    "comparative": {"type": "string"},
                    "third_person": {"type": "string"},
                    "second_person": {"type": "string"},
                    "first_person": {"type": "string"},
                },
                "additionalProperties": False,
            },
        },
        "guidelines": {
            "type": "array",
            "items": {
                "type": "object",
                "required": ["comparative", "third_person", "second_person", "first_person"],
                "properties": {
                    "comparative": {"type": "string"},
                    "third_person": {"type": "string"},
                    "second_person": {"type": "string"},
                    "first_person": {"type": "string"},
                },
                "additionalProperties": False,
            },
        }
    },
    "additionalProperties": False,
}


JUDGE_SCHEMA = {
    "type": "object",
    "required": [
        "score",
        "subscores",
        "major_missing_dimension",
        "strengths",
        "weaknesses",
        "revision_suggestions",
    ],
    "properties": {
        "score": {"type": "number"},
        "subscores": {
            "type": "object",
            "required": [
                "breadth",
                "source_faithfulness",
                "llm_behavioral_usefulness",
                "parallel_phrasing_quality",
                "clarity_nonredundancy",
                "edge_case_handling",
            ],
            "properties": {
                "breadth": {"type": "number"},
                "source_faithfulness": {"type": "number"},
                "llm_behavioral_usefulness": {"type": "number"},
                "parallel_phrasing_quality": {"type": "number"},
                "clarity_nonredundancy": {"type": "number"},
                "edge_case_handling": {"type": "number"},
            },
            "additionalProperties": False,
        },
        "major_missing_dimension": {"type": "boolean"},
        "strengths": {
            "type": "array",
            "items": {"type": "string"},
        },
        "weaknesses": {
            "type": "array",
            "items": {"type": "string"},
        },
        "revision_suggestions": {
            "type": "array",
            "items": {"type": "string"},
        },
    },
    "additionalProperties": False,
}

In [ ]:
def build_judge_rubric(constitutionName):
    return f"""
You are judging a candidate model constitution for {constitutionName}.

Score it from 0 to 100 using this rubric:

- Breadth of {constitutionName} coverage: 20 points
- Faithfulness to the foundational documents: 25 points
- Behavioral usefulness for an LLM: 20 points
- Quality of the four parallel phrasings: 15 points
- Clarity and non-redundancy: 10 points
- Edge-case/conflict handling: 10 points

Return valid JSON only with exactly this structure:

{{
  "score": 0,
  "subscores": {{
    "breadth": 0,
    "source_faithfulness": 0,
    "llm_behavioral_usefulness": 0,
    "parallel_phrasing_quality": 0,
    "clarity_nonredundancy": 0,
    "edge_case_handling": 0
  }},
  "major_missing_dimension": false,
  "strengths": ["..."],
  "weaknesses": ["..."],
  "revision_suggestions": ["..."]
}}
"""

In [ ]:
def build_judge_prompt(candidate_constitution, constitutionName, generation_prompt):
    return f"""
{build_judge_rubric(constitutionName)}

Original generation prompt:
{generation_prompt}

Candidate constitution:
{json.dumps(candidate_constitution, indent=2, ensure_ascii=False)}
"""

In [ ]:
def anonymize_critique_packet(critique_packet):
    """Model-facing view of the critique packet.

    Replaces the real judge identities with neutral labels ("Judge 1", "Judge 2", ...)
    under a FRESH RANDOM mapping on every call, and shuffles presentation order. This
    prevents a reviser from learning which model (or model family) produced a critique
    (a stable A/O/G-style scheme is learnable across items) and reduces position bias.

    Only used to build the revision prompt sent to a model. The saved packet
    (round_*_critique_packet.json) and everything a human reads keep the real names.
    """
    items = list(critique_packet)
    random.shuffle(items)
    anon = []
    for i, entry in enumerate(items, start=1):
        e = dict(entry)
        e["judge"] = f"Judge {i}"
        anon.append(e)
    return anon


def build_revision_prompt(current_constitution, critique_packet, round_number, constitutionName, generation_prompt):
    return f"""
You are a constitution writer and reviser.

You are revising the current working constitution for {constitutionName}.

Important instructions:
- Revise the current constitution only where doing so meaningfully improves it.
- Preserve strong existing content.
- Do not rewrite from scratch unless absolutely necessary.
- Do not make the constitution more generic.
- Do not add unnecessary length.
- Keep the same JSON structure.
- Preserve the four parallel phrasings:
  - comparative
  - third_person
  - second_person
  - first_person
- Make sure each set of four phrasings expresses the same underlying idea.
- Ground all content in the foundational documents.

Original generation prompt:
{generation_prompt}

Current working constitution:
{json.dumps(current_constitution, indent=2, ensure_ascii=False)}

Critique packet from judges:
{json.dumps(anonymize_critique_packet(critique_packet), indent=2, ensure_ascii=False)}

Round number:
{round_number}

Return valid JSON only with exactly this structure:

{{
  "overview": "...",
  "criteria": [
    {{
      "comparative": "Prefer the response that...",
      "third_person": "The model should...",
      "second_person": "You should...",
      "first_person": "I should..."
    }}
  ],
  "guidelines": [
    {{
      "comparative": "Prefer the response that...",
      "third_person": "The model should...",
      "second_person": "You should...",
      "first_person": "I should..."
    }}
  ]
}}
"""

In [ ]:
def generate_constitution(model_label, model_id, constitutionName, generation_prompt):
    system_prompt = (
        f"You are a constitution writer. "
        f"Your job is to write a candidate {constitutionName} model constitution."
    )

    raw = call_openrouter(
        model=model_id,
        system_prompt=system_prompt,
        user_prompt=generation_prompt,
        temperature=0.7,
    )

    obj = extract_json(raw)
    validate(instance=obj, schema=CONSTITUTION_SCHEMA)
    return obj


def judge_constitution(model_label, model_id, candidate_constitution, constitutionName, generation_prompt):
    system_prompt = (
        f"You are an impartial judge. "
        f"Your job is to evaluate {constitutionName} model constitutions using the provided rubric."
    )

    raw = call_openrouter(
        model=model_id,
        system_prompt=system_prompt,
        user_prompt=build_judge_prompt(candidate_constitution, constitutionName, generation_prompt),
        temperature=0.3,
    )

    obj = extract_json(raw)
    validate(instance=obj, schema=JUDGE_SCHEMA)
    return obj


def revise_constitution(
    model_label,
    model_id,
    current_constitution,
    critique_packet,
    round_number,
    constitutionName,
    generation_prompt,
    retries=3,
):
    system_prompt = (
        f"You are a constitution writer and reviser. "
        f"Your job is to revise a {constitutionName} constitution as part of a consensus process."
    )

    user_prompt = build_revision_prompt(
        current_constitution=current_constitution,
        critique_packet=critique_packet,
        round_number=round_number,
        constitutionName=constitutionName,
        generation_prompt=generation_prompt,
    )

    last_error = None

    for attempt in range(1, retries + 1):
        try:
            raw = call_openrouter(
                model=model_id,
                system_prompt=system_prompt,
                user_prompt=user_prompt,
                temperature=0.5,
                max_tokens=8000,
            )

            obj = extract_json(raw)
            validate(instance=obj, schema=CONSTITUTION_SCHEMA)
            return obj

        except Exception as e:
            last_error = e
            print(f"[ERROR] {model_label}Writer revision attempt {attempt}/{retries} failed: {e}")

            save_text(
                f"/content/debug_failed_{constitutionName}_{model_label}_round_{round_number}_attempt_{attempt}.txt",
                raw if "raw" in locals() and raw is not None else "NO RAW MODEL OUTPUT"
            )

            if attempt < retries:
                time.sleep(5)

    raise last_error

In [ ]:
def judge_all_models(candidate_constitution, constitutionName, generation_prompt):
    judgments = {}

    for label, model_id in MODELS.items():
        print(f"Judging with {label}Judge...")
        judgments[label] = judge_constitution(
            model_label=label,
            model_id=model_id,
            candidate_constitution=candidate_constitution,
            constitutionName = constitutionName,
            generation_prompt = generation_prompt
        )

    return judgments


def summarize_judgments(judgments):
    scores = [j["score"] for j in judgments.values()]

    return {
        "average_score": statistics.mean(scores),
        "min_score": min(scores),
        "max_score": max(scores),
        "major_missing_dimension": any(j["major_missing_dimension"] for j in judgments.values()), #if any modl thinks something is missing, marked as true
        "scores_by_judge": {
            label: judgment["score"]
            for label, judgment in judgments.items()
        },
    }


def build_critique_packet(judgments):
    packet = []

    for judge_label, judgment in judgments.items():
        packet.append({
            "judge": judge_label,
            "score": judgment["score"],
            "subscores": judgment["subscores"],
            "major_missing_dimension": judgment["major_missing_dimension"],
            "strengths": judgment["strengths"],
            "weaknesses": judgment["weaknesses"],
            "revision_suggestions": judgment["revision_suggestions"],
        })

    return packet


def passes_consensus(summary):
    return (
        summary["average_score"] >= 93
        and summary["min_score"] >= 90
        and not summary["major_missing_dimension"]
    )

In [ ]:
def run_pipeline(constitutionName, generation_prompt, max_rounds=3, output_dir="/content/constitution_results"):
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    run_dir = Path(output_dir) / f"{constitutionName}_{timestamp}"
    run_dir.mkdir(parents=True, exist_ok=True)

    save_text(run_dir / "generation_prompt.txt", generation_prompt)

    history = {
        "value_system": f"{constitutionName}",
        "run_dir": str(run_dir),
        "initial_candidates": {},
        "initial_candidate_summaries": {},
        "selected_initial_base": None,
        "rounds": [],
    }

    print("\n=== STEP 1: Initial writer generation ===")

    candidates = {}

    for label, model_id in MODELS.items():
        print(f"Generating with {label}Writer...")
        candidates[label] = generate_constitution(label, model_id, constitutionName, generation_prompt)
        save_json(run_dir / f"initial_candidate_{label}.json", candidates[label])

    history["initial_candidates"] = candidates

    print("\n=== STEP 2: Initial judging ===")

    candidate_judgments = {}
    candidate_summaries = {}

    for candidate_label, candidate_constitution in candidates.items():
        print(f"\nJudging initial candidate from {candidate_label}Writer...")

        judgments = judge_all_models(candidate_constitution, constitutionName, generation_prompt)
        summary = summarize_judgments(judgments)

        candidate_judgments[candidate_label] = judgments
        candidate_summaries[candidate_label] = summary

        save_json(run_dir / f"initial_judgments_for_{candidate_label}.json", judgments)
        save_json(run_dir / f"initial_summary_for_{candidate_label}.json", summary)

        print(f"{candidate_label} candidate summary:")
        print(summary)

    base_label = max(
        candidate_summaries,
        key=lambda label: candidate_summaries[label]["average_score"],
    )

    base_constitution = candidates[base_label]
    base_summary = candidate_summaries[base_label]

    history["initial_candidate_summaries"] = candidate_summaries
    history["selected_initial_base"] = base_label

    print(f"\nSelected base constitution: {base_label}Writer")
    print("Base summary:")
    print(base_summary)

    save_json(run_dir / "selected_initial_base.json", {
        "base_label": base_label,
        "base_summary": base_summary,
        "base_constitution": base_constitution,
    })

    writer_orders = [
        ["Gemini", "GPT", "Claude"],
        ["GPT", "Claude", "Gemini"],
        ["Claude", "Gemini", "GPT"],
    ]

    for round_number in range(1, max_rounds + 1):
        print(f"\n=== ROUND {round_number}: Critique current base ===")

        base_judgments = judge_all_models(base_constitution, constitutionName, generation_prompt)
        base_summary = summarize_judgments(base_judgments)
        critique_packet = build_critique_packet(base_judgments)

        save_json(run_dir / f"round_{round_number}_base_judgments.json", base_judgments)
        save_json(run_dir / f"round_{round_number}_base_summary.json", base_summary)
        save_json(run_dir / f"round_{round_number}_critique_packet.json", critique_packet)

        print("Current base summary:")
        print(base_summary)
        MIN_REVISIONS = 1
        if round_number > MIN_REVISIONS and passes_consensus(base_summary):
            print("\nConsensus threshold met. Stopping.")
            break

        print(f"\n=== ROUND {round_number}: Sequential writer revision ===")

        revised_constitution = base_constitution
        writer_order = writer_orders[(round_number - 1) % len(writer_orders)]

        for writer_label in writer_order:
            print(f"Revising with {writer_label}Writer...")

            revised_constitution = revise_constitution(
                model_label=writer_label,
                model_id=MODELS[writer_label],
                current_constitution=revised_constitution,
                critique_packet=critique_packet,
                round_number=round_number,
                constitutionName = constitutionName,
                generation_prompt = generation_prompt
            )

            save_json(
                run_dir / f"round_{round_number}_after_{writer_label}Writer.json",
                revised_constitution,
            )

        print(f"\n=== ROUND {round_number}: Judge revised constitution ===")

        revised_judgments = judge_all_models(revised_constitution, constitutionName, generation_prompt)
        revised_summary = summarize_judgments(revised_judgments)

        save_json(run_dir / f"round_{round_number}_revised_judgments.json", revised_judgments)
        save_json(run_dir / f"round_{round_number}_revised_summary.json", revised_summary)

        print("Revised summary:")
        print(revised_summary)

        base_passes = passes_consensus(base_summary)
        revised_passes = passes_consensus(revised_summary)

        improved = (
            (revised_passes and not base_passes)
            or revised_summary["average_score"] > base_summary["average_score"]
            or (
                revised_summary["average_score"] == base_summary["average_score"]
                and revised_summary["min_score"] > base_summary["min_score"]
            )
            or (
                base_summary["major_missing_dimension"]
                and not revised_summary["major_missing_dimension"]
            )
        )

        round_record = {
            "round_number": round_number,
            "writer_order": writer_order,
            "base_summary": base_summary,
            "revised_summary": revised_summary,
            "accepted_revision": improved,
        }

        history["rounds"].append(round_record)
        save_json(run_dir / f"round_{round_number}_record.json", round_record)

        if improved:
            print("Accepted revised constitution as new base.")
            base_constitution = revised_constitution
            base_summary = revised_summary
        else:
            print("Rejected revised constitution. Keeping previous base.")

        if round_number >= MIN_REVISIONS and passes_consensus(base_summary):
            print("\nConsensus threshold met after revision. Stopping.")
            break

    print("\n=== FINAL JUDGING ===")

    final_judgments = judge_all_models(base_constitution, constitutionName, generation_prompt)
    final_summary = summarize_judgments(final_judgments)

    history["final_constitution"] = base_constitution
    history["final_judgments"] = final_judgments
    history["final_summary"] = final_summary

    save_json(run_dir / "final_constitution.json", base_constitution)
    save_json(run_dir / "final_judgments.json", final_judgments)
    save_json(run_dir / "final_summary.json", final_summary)
    save_json(run_dir / "run_history.json", history)

    print("\nFinal summary:")
    print(final_summary)
    print(f"\nSaved results to: {run_dir}")

    return history

In [ ]:
def run_everything(constitutionName, docs_root, max_rounds):
    docs_folder = Path(docs_root) / f"{constitutionName} Docs"
    foundationalDocs = load_markdown_docs(str(docs_folder))
    generation_prompt = generateGenerationPrompt(constitutionName, foundationalDocs)

    return run_pipeline(
        constitutionName=constitutionName,
        generation_prompt=generation_prompt,
        max_rounds=max_rounds,
        output_dir=f"/content/{constitutionName}_constitution_results",
    )

In [ ]:
Hinduism = run_everything("Hinduism", "/content", 3)


=== STEP 1: Initial writer generation ===
Generating with GPTWriter...
Generating with ClaudeWriter...
Generating with GeminiWriter...

=== STEP 2: Initial judging ===

Judging initial candidate from GPTWriter...
Judging with GPTJudge...
Judging with ClaudeJudge...
Judging with GeminiJudge...
GPT candidate summary:
{'average_score': 91, 'min_score': 84, 'max_score': 99, 'major_missing_dimension': False, 'scores_by_judge': {'GPT': 90, 'Claude': 84, 'Gemini': 99}}

Judging initial candidate from ClaudeWriter...
Judging with GPTJudge...
Judging with ClaudeJudge...
Judging with GeminiJudge...
Claude candidate summary:
{'average_score': 91, 'min_score': 87, 'max_score': 98, 'major_missing_dimension': False, 'scores_by_judge': {'GPT': 88, 'Claude': 87, 'Gemini': 98}}

Judging initial candidate from GeminiWriter...
Judging with GPTJudge...
Judging with ClaudeJudge...
Judging with GeminiJudge...
Gemini candidate summary:
{'average_score': 76.66666666666667, 'min_score': 62, 'max_score': 96, '

In [ ]:
Daoism = run_everything("Daoism", "/content", 3)


=== STEP 1: Initial writer generation ===
Generating with GPTWriter...
Generating with ClaudeWriter...
Generating with GeminiWriter...

=== STEP 2: Initial judging ===

Judging initial candidate from GPTWriter...
Judging with GPTJudge...
Judging with ClaudeJudge...
Judging with GeminiJudge...
GPT candidate summary:
{'average_score': 93.66666666666667, 'min_score': 87, 'max_score': 100, 'major_missing_dimension': False, 'scores_by_judge': {'GPT': 94, 'Claude': 87, 'Gemini': 100}}

Judging initial candidate from ClaudeWriter...
Judging with GPTJudge...
Judging with ClaudeJudge...
Judging with GeminiJudge...
Claude candidate summary:
{'average_score': 94.66666666666667, 'min_score': 90, 'max_score': 100, 'major_missing_dimension': False, 'scores_by_judge': {'GPT': 94, 'Claude': 90, 'Gemini': 100}}

Judging initial candidate from GeminiWriter...
Judging with GPTJudge...
Judging with ClaudeJudge...
Judging with GeminiJudge...
Gemini candidate summary:
{'average_score': 89.33333333333333, '